# Phase 18 — Target Comparison & Research Direction Assessment

**Project:** Australian Stock Market Analytics & Return Modeling

## Objective
Compare a small set of clearly defined research targets using the existing Phase 13 chronological splits and the five audited candidate predictors. The aim is to determine whether raw 3-month return, benchmark-relative 3-month return, or a simple positive-return classification target is the most defensible next research direction.

**Important safeguards**
- Do not tune or select a final model using test results.
- This is a research-direction assessment, not a claim of investment profitability.
- Benchmark-relative targets are used only if the benchmark columns exist and are correctly aligned.
- The notebook reports limitations if required fields are unavailable.

## 1. Setup and locate Phase 13 split files

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, precision_score, recall_score
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
RANDOM_STATE = 42

TRAIN_REL = Path("data/processed/model_features/asx_3m_strict_split_train.csv")
VALID_REL = Path("data/processed/model_features/asx_3m_strict_split_validation.csv")
TEST_REL = Path("data/processed/model_features/asx_3m_strict_split_test.csv")

def locate_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if all((p / r).exists() for r in [TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    for p in start.iterdir() if start.exists() else []:
        if p.is_dir() and all((p / r).exists() for r in [TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    return None

PROJECT_ROOT = locate_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find Phase 13 split files. Run Phase 13 first.")

def read_split(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip().lower() for c in df.columns]
    for c in ["date", "target_endpoint_date_3m"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["forward_return_3m"] = pd.to_numeric(df["forward_return_3m"], errors="coerce")
    return df

train = read_split(PROJECT_ROOT / TRAIN_REL)
valid = read_split(PROJECT_ROOT / VALID_REL)
test = read_split(PROJECT_ROOT / TEST_REL)

print("Project root:", PROJECT_ROOT)
print("Rows:", {"train": len(train), "validation": len(valid), "test": len(test)})

Project root: e:\Projects\Australian Stock Market Data Science Project
Rows: {'train': 40447, 'validation': 6084, 'test': 9412}


## 2. Verify splits and identify available predictors/targets

In [2]:
for name, part in [("train", train), ("validation", valid), ("test", test)]:
    if part.empty:
        raise ValueError(f"{name} split is empty.")
    required = {"ticker", "date", "target_endpoint_date_3m", "forward_return_3m"}
    missing = required - set(part.columns)
    if missing:
        raise ValueError(f"{name} split missing: {sorted(missing)}")

assert train["target_endpoint_date_3m"].max() < valid["date"].min()
assert valid["target_endpoint_date_3m"].max() < test["date"].min()

candidate_features = [
    "momentum_21d", "momentum_63d", "benchmark_return_1d",
    "benchmark_volatility_21d", "benchmark_volatility_63d"
]
feature_cols = [c for c in candidate_features if c in train.columns]
if not feature_cols:
    raise ValueError("No candidate predictors found.")

for part in [train, valid, test]:
    for c in feature_cols:
        part[c] = pd.to_numeric(part[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    part["forward_return_3m"] = pd.to_numeric(part["forward_return_3m"], errors="coerce")

benchmark_target_candidates = [
    "benchmark_forward_return_3m",
    "forward_benchmark_return_3m",
    "benchmark_forward_return",
    "benchmark_return_63d_forward"
]
benchmark_target_col = next(
    (c for c in benchmark_target_candidates if c in train.columns), None
)
print("Predictors:", feature_cols)
print("Benchmark forward-return target available:", benchmark_target_col)
print("Training target summary:")
display(train["forward_return_3m"].describe())

Predictors: ['momentum_21d', 'momentum_63d', 'benchmark_return_1d', 'benchmark_volatility_21d', 'benchmark_volatility_63d']
Benchmark forward-return target available: None
Training target summary:


count    40447.000000
mean         0.021461
std          0.173414
min         -0.837209
25%         -0.074401
50%          0.015444
75%          0.103177
max          1.678061
Name: forward_return_3m, dtype: float64

## 3. Establish target definitions

- **Raw return regression:** existing 63-observation forward return.
- **Positive-return classification:** 1 if the forward return is strictly greater than zero, otherwise 0.
- **Benchmark-relative regression:** available only if a verified benchmark forward-return target exists in the split data. We do not infer it from same-day benchmark returns.

In [3]:
for part in [train, valid, test]:
    part["positive_forward_return"] = (part["forward_return_3m"] > 0).astype("int64")

print("Positive-return rates:")
display(pd.DataFrame([
    {"split": name, "rows": len(part), "positive_rate": part["positive_forward_return"].mean(),
     "negative_or_zero_rate": (part["positive_forward_return"] == 0).mean()}
    for name, part in [("train", train), ("validation", valid), ("test", test)]
]))

Positive-return rates:


,split,rows,positive_rate,negative_or_zero_rate
0,train,40447,0.544119,0.455881
1,validation,6084,0.659928,0.340072
2,test,9412,0.472694,0.527306


## 4. Evaluate raw-return regression candidates

Fit preprocessing on training data only. Select the regression model using validation MAE. Test metrics are reported once for the selected validation winner and the mean baseline.

In [4]:
reg_models = {
    "Training mean baseline": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="mean"))
    ]),
    "Ridge Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=250, min_samples_leaf=20, max_features=0.8,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

X_train, y_train = train[feature_cols], train["forward_return_3m"].astype(float)
X_valid, y_valid = valid[feature_cols], valid["forward_return_3m"].astype(float)
X_test, y_test = test[feature_cols], test["forward_return_3m"].astype(float)

reg_fitted, reg_val_rows = {}, []
for name, model in reg_models.items():
    model.fit(X_train, y_train)
    reg_fitted[name] = model
    pv = model.predict(X_valid)
    reg_val_rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_valid, pv),
        "RMSE": np.sqrt(mean_squared_error(y_valid, pv)),
        "R2": r2_score(y_valid, pv),
        "validation_predicted_mean": np.mean(pv)
    })
reg_validation = pd.DataFrame(reg_val_rows).sort_values("MAE").reset_index(drop=True)
display(reg_validation)

,model,MAE,RMSE,R2,validation_predicted_mean
0,Training mean baseline,0.141984,0.195113,-0.081749,0.021461
1,Random Forest,0.146553,0.204755,-0.191312,0.030595
2,Ridge Regression,0.147386,0.202863,-0.169392,0.011043


## 5. Evaluate positive-return classification candidates

This is a separate target definition, not a substitute for regression. Class imbalance and probability calibration matter, so we report balanced accuracy, F1, and ROC-AUC when defined. Model selection uses validation balanced accuracy; test results are only for the selected validation winner.

In [5]:
clf_models = {
    "Majority-class baseline": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest Classifier": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=20, max_features=0.8,
            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

yc_train = train["positive_forward_return"].astype(int)
yc_valid = valid["positive_forward_return"].astype(int)
yc_test = test["positive_forward_return"].astype(int)

clf_fitted, clf_val_rows = {}, []
for name, model in clf_models.items():
    model.fit(X_train, yc_train)
    clf_fitted[name] = model
    pv = model.predict(X_valid)
    row = {
        "model": name,
        "balanced_accuracy": balanced_accuracy_score(yc_valid, pv),
        "F1_positive_class": f1_score(yc_valid, pv, zero_division=0),
        "accuracy": accuracy_score(yc_valid, pv)
    }
    if len(np.unique(yc_valid)) == 2 and hasattr(model, "predict_proba"):
        try:
            row["ROC_AUC"] = roc_auc_score(yc_valid, model.predict_proba(X_valid)[:, 1])
        except ValueError:
            row["ROC_AUC"] = np.nan
    else:
        row["ROC_AUC"] = np.nan
    clf_val_rows.append(row)

clf_validation = pd.DataFrame(clf_val_rows).sort_values(
    ["balanced_accuracy", "F1_positive_class"], ascending=False
).reset_index(drop=True)
display(clf_validation)

,model,balanced_accuracy,F1_positive_class,accuracy,ROC_AUC
0,Random Forest Classifier,0.515283,0.599448,0.523340,0.520406
1,Majority-class baseline,0.500000,0.795128,0.659928,0.500000
2,Logistic Regression,0.488728,0.586707,0.503287,0.488505


## 6. One-time held-out evaluation of validation-selected models

In [6]:
best_reg_name = str(reg_validation.iloc[0]["model"])
best_clf_name = str(clf_validation.iloc[0]["model"])

def reg_metrics(y_true, y_pred):
    return {
        "n": len(y_true),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "actual_mean": np.mean(y_true),
        "predicted_mean": np.mean(y_pred)
    }

reg_test_rows = []
for name in ["Training mean baseline", best_reg_name]:
    model = reg_fitted[name]
    pt = model.predict(X_test)
    reg_test_rows.append({"model": name, **reg_metrics(y_test, pt)})
reg_test = pd.DataFrame(reg_test_rows).drop_duplicates(subset=["model"])
display(reg_test)

clf_model = clf_fitted[best_clf_name]
pred_class = clf_model.predict(X_test)
class_test = {
    "model": best_clf_name,
    "n": len(yc_test),
    "accuracy": accuracy_score(yc_test, pred_class),
    "balanced_accuracy": balanced_accuracy_score(yc_test, pred_class),
    "F1_positive_class": f1_score(yc_test, pred_class, zero_division=0),
    "positive_rate_actual": float(yc_test.mean()),
    "positive_rate_predicted": float(pred_class.mean())
}
if len(np.unique(yc_test)) == 2 and hasattr(clf_model, "predict_proba"):
    try:
        class_test["ROC_AUC"] = roc_auc_score(yc_test, clf_model.predict_proba(X_test)[:, 1])
    except ValueError:
        class_test["ROC_AUC"] = None
else:
    class_test["ROC_AUC"] = None

display(pd.DataFrame([class_test]))

,model,n,MAE,RMSE,R2,actual_mean,predicted_mean
0,Training mean baseline,9412,0.142559,0.183638,-0.030017,-0.009888,0.021461


,model,n,accuracy,balanced_accuracy,F1_positive_class,positive_rate_actual,positive_rate_predicted,ROC_AUC
0,Random Forest Classifier,9412,0.485869,0.475677,0.347052,0.472694,0.314705,0.453125


## 7. Optional benchmark-relative target check

If a verified benchmark forward-return column exists, evaluate its coverage and scale. This notebook does not automatically train a relative-return model unless the target is present and non-missing in every split.

In [7]:
benchmark_target_assessment = {
    "available": benchmark_target_col is not None,
    "column": benchmark_target_col,
    "status": "not available; do not infer benchmark forward return from contemporaneous benchmark features"
}
if benchmark_target_col:
    coverage = []
    for name, part in [("train", train), ("validation", valid), ("test", test)]:
        vals = pd.to_numeric(part[benchmark_target_col], errors="coerce")
        coverage.append({
            "split": name, "rows": len(part),
            "non_missing_target_rows": int(vals.notna().sum()),
            "mean": float(vals.mean()) if vals.notna().any() else np.nan,
            "std": float(vals.std()) if vals.notna().any() else np.nan
        })
    display(pd.DataFrame(coverage))
    benchmark_target_assessment["status"] = "column found; inspect coverage and confirm endpoint alignment before modeling"

## 8. Save research-direction assessment

In [8]:
OUT = PROJECT_ROOT / "reports" / "research_direction_assessment"
OUT.mkdir(parents=True, exist_ok=True)

reg_validation.to_csv(OUT / "raw_return_regression_validation.csv", index=False)
clf_validation.to_csv(OUT / "positive_return_classification_validation.csv", index=False)
reg_test.to_csv(OUT / "raw_return_regression_test_selected.csv", index=False)
pd.DataFrame([class_test]).to_csv(OUT / "positive_return_classification_test_selected.csv", index=False)

best_reg_val_mae = float(reg_validation.iloc[0]["MAE"])
baseline_reg_val_mae = float(reg_validation.loc[
    reg_validation["model"] == "Training mean baseline", "MAE"
].iloc[0])
best_clf_bal_acc = float(clf_validation.iloc[0]["balanced_accuracy"])
majority_bal_acc = float(clf_validation.loc[
    clf_validation["model"] == "Majority-class baseline", "balanced_accuracy"
].iloc[0])

summary = {
    "phase": 18,
    "target_rows": {"train": len(train), "validation": len(valid), "test": len(test)},
    "features": feature_cols,
    "raw_return_regression": {
        "best_validation_model": best_reg_name,
        "best_validation_mae": best_reg_val_mae,
        "baseline_validation_mae": baseline_reg_val_mae,
        "best_beats_baseline_on_validation_mae": bool(best_reg_val_mae < baseline_reg_val_mae),
        "selected_test_results": reg_test.to_dict(orient="records")
    },
    "positive_return_classification": {
        "best_validation_model": best_clf_name,
        "best_validation_balanced_accuracy": best_clf_bal_acc,
        "majority_baseline_validation_balanced_accuracy": majority_bal_acc,
        "best_beats_baseline_on_validation_balanced_accuracy": bool(best_clf_bal_acc > majority_bal_acc),
        "selected_test_results": class_test
    },
    "benchmark_relative_target": benchmark_target_assessment,
    "warning": "Exploratory research only. Classification of positive returns is not a trading strategy or profitability estimate."
}
with open(OUT / "phase18_research_direction_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print(json.dumps(summary, indent=2, default=str))
print("\nOutputs saved to:", OUT)

{
  "phase": 18,
  "target_rows": {
    "train": 40447,
    "validation": 6084,
    "test": 9412
  },
  "features": [
    "momentum_21d",
    "momentum_63d",
    "benchmark_return_1d",
    "benchmark_volatility_21d",
    "benchmark_volatility_63d"
  ],
  "raw_return_regression": {
    "best_validation_model": "Training mean baseline",
    "best_validation_mae": 0.14198443955103976,
    "baseline_validation_mae": 0.14198443955103976,
    "best_beats_baseline_on_validation_mae": false,
    "selected_test_results": [
      {
        "model": "Training mean baseline",
        "n": 9412,
        "MAE": 0.14255885547150116,
        "RMSE": 0.18363815182311155,
        "R2": -0.030017160524183906,
        "actual_mean": -0.009887806217044012,
        "predicted_mean": 0.021461323363067605
      }
    ]
  },
  "positive_return_classification": {
    "best_validation_model": "Random Forest Classifier",
    "best_validation_balanced_accuracy": 0.5152825286037678,
    "majority_baseline_validatio

## 9. Completion checklist

- Review validation results before interpreting test metrics.
- Check positive-class prevalence; accuracy alone can be misleading.
- Confirm the benchmark-relative target is not available unless a properly aligned column is present.
- Do not claim trading profitability without a separate backtest with costs, execution assumptions, and portfolio constraints.